# **06. PREDICTIVE MODEL**

## **Greater London house price forecast**
La notebook se encarga de documentar el proceso de creacion y evaluacion de un modelo predictivo, `prophet`, para realizar una prediccion de la evolucion de las viviendas en el area metropolitana de Londres.

### **Objetivo**
Visualizar los datos disponibles en la tabla Silver, para realizar transformaciones que aporten valor al dataset, guardándose en la última capa (gold).

### **Tabla de entrada**
`workspace.default.london_gold` — 2,384,979 filas · 22 columnas


In [0]:
%pip install prophet

In [0]:
df_pred = spark.table("workspace.default.london_gold")
display(df_pred.limit(5))

In [0]:
import pandas as pd
import pyspark.sql.functions as F

# 1. Agregación a nivel mensual usando PySpark (¡Evita el colapso de memoria!)
# Agrupamos por 'year' y 'month', y calculamos el precio medio usando la columna 'price'
df_monthly_spark = df_pred.groupBy("year", "month") \
                          .agg(F.mean("price").alias("avg_price")) \
                          .orderBy("year", "month")

# 2. Ahora sí, pasamos el resultado (que serán solo unas pocas decenas de filas) a Pandas
df_pandas = df_monthly_spark.toPandas()

# 3. Crear la columna de fecha 'ds' que exige Prophet
df_pandas['ds'] = pd.to_datetime(df_pandas[['year', 'month']].assign(day=1))

# 4. Seleccionar las columnas finales y renombrar el objetivo a 'y'
df_prophet = df_pandas[['ds', 'avg_price']].rename(columns={'avg_price': 'y'})

# 5. Split de Train y Test (Mejora profesional: reservamos los últimos 12 meses)
test_months = 12
df_train = df_prophet.iloc[:-test_months]
df_test = df_prophet.iloc[-test_months:]

print("✅ Agregación en PySpark y conversión a Pandas completada.")
print(f"📊 Datos de entrenamiento (Train): {len(df_train)} meses listos para el modelo.")
print(f"🎯 Datos de validación (Test): {len(df_test)} meses reservados para medir el error.")

In [0]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Instanciar el modelo con la configuración del briefing
# Desactivamos estacionalidades diarias/semanales al ser datos mensuales agregados
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05 # Controla qué tan rápido cambia la tendencia
)

# 2. Entrenar el modelo SÓLO con el conjunto de entrenamiento
model.fit(df_train)
print("✅ Modelo entrenado con los datos históricos (Train).")

# 3. Preparar el dataframe de fechas para la predicción
# Le pasamos a Prophet las fechas ('ds') de los 12 meses que ocultamos
future_test = df_test[['ds']]

# 4. Generar la predicción para ese periodo de prueba
forecast_test = model.predict(future_test)

# 5. Calcular el error real del modelo (Métricas)
y_true = df_test['y'].values
y_pred = forecast_test['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"📊 Resultados de la Validación (Últimos 12 meses):")
print(f"   - Error Absoluto Medio (MAE): £{mae:,.2f}")
print(f"   - Raíz del Error Cuadrático Medio (RMSE): £{rmse:,.2f}")

# 6. Crear una tabla comparativa rápida para verla en Databricks
comparativa = df_test[['ds', 'y']].rename(columns={'y': 'precio_real'})
comparativa['precio_predicho'] = y_pred.round(2)
comparativa['error_libras'] = (comparativa['precio_real'] - comparativa['precio_predicho']).round(2)

display(comparativa)

In [0]:
import matplotlib.pyplot as plt
from prophet import Prophet

# 1. Entrenar el modelo final con TODOS los datos disponibles
modelo_final = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)

# Usamos df_prophet, que contiene el histórico completo sin recortes
modelo_final.fit(df_prophet) 
print("✅ Modelo final entrenado con el 100% de los datos históricos.")

# 2. Crear las fechas futuras: 5 años = 60 meses
futuro = modelo_final.make_future_dataframe(periods=60, freq='MS')

# 3. Generar la predicción
prediccion = modelo_final.predict(futuro)

# 4. Visualización profesional (basada en el briefing)
fig, ax = plt.subplots(figsize=(14, 6))

# Dibujar datos históricos
ax.plot(df_prophet['ds'], df_prophet['y'], 
        color='#1B2A4A', label='Precio histórico', linewidth=1.5)

# Separar solo la parte futura para dibujarla en otro color
solo_futuro = prediccion[prediccion['ds'] > df_prophet['ds'].max()]

# Dibujar la línea de predicción futura
ax.plot(solo_futuro['ds'], solo_futuro['yhat'], 
        color='#E8A020', linestyle='--', label='Predicción 2025-2029', linewidth=2)

# Añadir el intervalo de confianza (la "sombra" alrededor de la predicción)
ax.fill_between(solo_futuro['ds'], 
                solo_futuro['yhat_lower'], solo_futuro['yhat_upper'], 
                alpha=0.2, color='#E8A020', label='Intervalo de confianza')

# Estilos y etiquetas
ax.set_title('Greater London House Prices — Historical & Forecast (5 Years)')
ax.set_xlabel('Año')
ax.set_ylabel('Precio Medio (£)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

# Mostrar el gráfico en Databricks
plt.show()

# Opcional: mostrar los últimos meses de la tabla de predicción
display(prediccion[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())